# Assignment 18 — Python Code Analysis and Correction with CrewAI

**Objective:** Use a CrewAI agent system to analyze and correct errors in Python code. Two agents do the work — a **Code Analyzer** that finds syntax and logical errors and a **Code Corrector** that fixes them — overseen by a **Manager** agent. Tasks run with `Process.sequential` and the Crew has **planning enabled** (`planning=True`).

**Framework:** CrewAI `1.15.33` (current release), installed with **uv**.

| Assignment concept | Where it is in this notebook |
|---|---|
| Code Analyzer agent (identifies syntax + logical errors) | Section 6 `code_analyzer` |
| Code Corrector agent (fixes identified errors) | Section 6 `code_corrector` |
| Manager agent (oversees analysis and correction) | Section 6 `manager`, attached to the Crew as `manager_agent` |
| Analysis Task → Correction Task, sequential | Section 7 tasks, Section 8 `Crew(process=Process.sequential)` |
| `CodeInterpreterTool` used by the Code Analyzer to evaluate the code | Section 5 (tool definition and self-test), given to `code_analyzer` in Section 6 |
| Planning enabled | Section 8 `Crew(planning=True, planning_llm=...)` |
| Expected output: list of errors, then corrected code | Section 9 results, Section 10 deterministic verification |

> **Note on `CodeInterpreterTool`.** The built-in `crewai_tools.CodeInterpreterTool` (Docker-based) was removed in the CrewAI 1.15 line; `Agent(allow_code_execution=True)` is now a deprecated no-op that says *"CodeInterpreterTool is no longer available"*. This notebook therefore defines a small `CodeInterpreterTool` on `crewai.tools.BaseTool` that runs the submitted Python in a separate interpreter process with a timeout. Same name, same job, no Docker needed in Colab.


## 1. Install dependencies
Installed with **uv** (fast). The version is pinned so the notebook stays reproducible. Only `crewai` is needed; the tool below is built on `crewai.tools.BaseTool`, which ships with it.

In [1]:
import sys
%pip install -q uv
!uv pip install -q --python "{sys.executable}" "crewai==1.15.33"

import crewai
print("CrewAI version:", crewai.__version__)

CrewAI version: 1.15.33


## 2. API key
Looks for the key in **Colab Secrets** first (`OPENAI_API_KEY`), then an environment variable, then asks you to paste it. The key is never printed.

In [2]:
import os
from getpass import getpass


def load_api_key() -> str:
    try:
        from google.colab import userdata  # only exists inside Colab
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        pass  # not in Colab, secret missing, or notebook access not granted
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key
    return getpass("OPENAI_API_KEY not found in Colab Secrets or environment. Paste your key: ")


API_KEY = load_api_key().strip()
if not API_KEY:
    raise ValueError("No API key provided. Add OPENAI_API_KEY to Colab Secrets or paste it when prompted.")
os.environ["OPENAI_API_KEY"] = API_KEY
print("API key loaded.")

API key loaded.


## 3. Settings and model
All settings live here. The first reachable model in `MODEL_CANDIDATES` is used. CrewAI's optional trace upload and telemetry are switched off so nothing is sent anywhere except the OpenAI API and no first-run prompt can stall the notebook.

In [3]:
import os
from openai import OpenAI
from crewai import LLM

MODEL_CANDIDATES = ["gpt-4o-mini", "gpt-4o"]
TOOL_TIMEOUT_SECONDS = 10      # hard limit for one code-interpreter run
MAX_CODE_CHARS = 20_000        # input validation limit

os.environ["CREWAI_TRACING_ENABLED"] = "false"   # no trace upload / no first-run prompt
os.environ["OTEL_SDK_DISABLED"] = "true"          # no anonymous telemetry
os.environ["CREWAI_DISABLE_VERSION_CHECK"] = "true"

client = OpenAI(api_key=API_KEY)
MODEL = None
for candidate in MODEL_CANDIDATES:
    try:
        client.models.retrieve(candidate)
        MODEL = candidate
        break
    except Exception as exc:  # wrong key, no access to this model, network
        print(f"{candidate}: not reachable ({type(exc).__name__})")
if MODEL is None:
    raise RuntimeError("None of MODEL_CANDIDATES is reachable with this key. Check the key or edit the list.")

llm = LLM(model=MODEL, temperature=0)
print("Using model:", MODEL)

Using model: gpt-4o-mini


## 4. Input code
The buggy function from the assignment, exactly as given (flat indentation inside the `for` loop, no handling of `n == 0`). Inputs are validated before anything is sent to the model: must be a non-empty string within the size limit. The self-test proves the validation rejects bad input.

In [4]:
BUGGY_CODE = """
def fibonacci_iterative(n):
    if n < 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]

    fib_sequence = [0, 1]
    for i in range(2, n):
    next_fib = fib_sequence[-1] + fib_sequence[-2]
    fib_sequence.append(next_fib)
    return fib_sequence
"""

SPEC = ("fibonacci_iterative(n) must return a list of the first n Fibonacci numbers "
        "starting 0, 1, 1, 2, ... For n <= 0 it must return an empty list.")


def validate_code(source) -> str:
    """Trust boundary: the code is untrusted text. Check type, emptiness and size."""
    if not isinstance(source, str):
        raise ValueError(f"code must be a string, got {type(source).__name__}")
    if not source.strip():
        raise ValueError("code is empty")
    if len(source) > MAX_CODE_CHARS:
        raise ValueError(f"code is too long ({len(source)} chars > {MAX_CODE_CHARS})")
    return source.strip("\n")


# should-fail self-test
for bad in ["", "   \n", 123, "x" * (MAX_CODE_CHARS + 1)]:
    try:
        validate_code(bad)
        raise AssertionError(f"validation accepted bad input: {bad!r:.30}")
    except ValueError:
        pass

CODE_TO_FIX = validate_code(BUGGY_CODE)
print("validation self-test: PASS")
print(CODE_TO_FIX)

validation self-test: PASS
def fibonacci_iterative(n):
    if n < 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]

    fib_sequence = [0, 1]
    for i in range(2, n):
    next_fib = fib_sequence[-1] + fib_sequence[-2]
    fib_sequence.append(next_fib)
    return fib_sequence


## 5. CodeInterpreterTool
Runs a Python snippet in a **separate interpreter process** (`python -I`, isolated mode) with a timeout and returns stdout, stderr and the exit code. The Code Analyzer uses it to *evaluate* the code rather than guess: a syntax error shows up as a traceback, a logical error shows up as wrong output when the agent calls the function. The self-test runs the buggy code and confirms the tool reports the `IndentationError`.

In [5]:
import subprocess
import sys
import tempfile
from pathlib import Path

from crewai.tools import BaseTool
from pydantic import BaseModel, Field


class CodeInterpreterInput(BaseModel):
    code: str = Field(description="Complete Python 3 source code to execute.")


class CodeInterpreterTool(BaseTool):
    name: str = "CodeInterpreterTool"
    description: str = (
        "Executes Python 3 code in a separate process and returns stdout, stderr and the exit code. "
        "Use it to check whether code compiles and to run the function with test inputs. "
        "Pass the complete code including any print(...) calls you want to see."
    )
    args_schema: type[BaseModel] = CodeInterpreterInput
    timeout_seconds: int = TOOL_TIMEOUT_SECONDS

    def _run(self, code: str) -> str:
        if not isinstance(code, str) or not code.strip():
            return "ERROR: no code provided."
        if len(code) > MAX_CODE_CHARS:
            return f"ERROR: code too long ({len(code)} chars > {MAX_CODE_CHARS})."
        with tempfile.TemporaryDirectory() as tmp:
            script = Path(tmp) / "snippet.py"
            script.write_text(code, encoding="utf-8")
            try:
                proc = subprocess.run(
                    [sys.executable, "-I", str(script)],
                    capture_output=True, text=True, timeout=self.timeout_seconds, cwd=tmp,
                )
            except subprocess.TimeoutExpired:
                return f"ERROR: execution exceeded {self.timeout_seconds}s and was stopped."
        out = proc.stdout.strip()[-4000:]
        err = proc.stderr.strip()[-4000:]
        return f"exit code: {proc.returncode}\nstdout:\n{out or '(empty)'}\nstderr:\n{err or '(empty)'}"


code_interpreter = CodeInterpreterTool()

# self-test 1: a correct snippet runs
assert "exit code: 0" in code_interpreter.run(code="print(1 + 1)")
# self-test 2 (should-fail path): the buggy code must NOT run cleanly
report = code_interpreter.run(code=CODE_TO_FIX + "\nprint(fibonacci_iterative(10))")
assert "exit code: 0" not in report and "IndentationError" in report, report
# self-test 3: an infinite loop is stopped by the timeout
assert "exceeded" in CodeInterpreterTool(timeout_seconds=2).run(code="while True: pass")
print("tool self-test: PASS\n")
print(report)

tool self-test: PASS

exit code: 1
stdout:
(empty)
stderr:
File "/tmp/tmpxy2r4zgz/snippet.py", line 11
    next_fib = fib_sequence[-1] + fib_sequence[-2]
    ^^^^^^^^
IndentationError: expected an indented block after 'for' statement on line 10


## 6. Agents
Three agents. Only the Code Analyzer gets the `CodeInterpreterTool`; the Corrector works from the analysis it receives through task `context`. The Manager has no task of its own: it is registered on the Crew as `manager_agent`, the same way the course's fraud-detection example does it. Be clear about what that means: with `Process.sequential` CrewAI itself runs the tasks in the given order and the manager does not delegate; oversight comes from the planning step in Section 8, which drafts a plan for every task before execution. Switching one line to `Process.hierarchical` makes the manager actively hand the tasks to the two agents (see the comment in Section 8).

In [6]:
from crewai import Agent

code_analyzer = Agent(
    role="Code Analyzer",
    goal="Identify every syntax error and logical error in the Python code you are given.",
    backstory=(
        "A meticulous senior Python reviewer. You never guess: you run the code with the "
        "CodeInterpreterTool, read the traceback, then test the function against its specification "
        "with several inputs to expose logical errors."
    ),
    tools=[code_interpreter],
    llm=llm,
    max_iter=8,
    verbose=False,
)

code_corrector = Agent(
    role="Code Corrector",
    goal="Produce a corrected version of the Python code that fixes every identified error.",
    backstory=(
        "A pragmatic Python engineer. You make the smallest set of changes that fixes the reported "
        "errors, keep the original function name and style, and return complete runnable code."
    ),
    llm=llm,
    max_iter=5,
    verbose=False,
)

manager = Agent(
    role="Code Review Manager",
    goal="Make sure the analysis is complete before correction starts and the final code satisfies the specification.",
    backstory="Oversees the code review workflow and keeps the analysis and correction steps on track.",
    llm=llm,
    allow_delegation=True,
    verbose=False,
)
print("agents ready:", [a.role for a in (code_analyzer, code_corrector, manager)])

agents ready: ['Code Analyzer', 'Code Corrector', 'Code Review Manager']


## 7. Tasks
The **Analysis Task** asks for a numbered list of errors (type, line, why, fix). The **Correction Task** receives that list via `context=[analysis_task]` and must answer with one ```` ```python ```` block containing the whole corrected function, followed by the list of changes. The fenced block is what Section 10 extracts and tests.

In [7]:
from crewai import Task

analysis_task = Task(
    description=(
        "Analyze the following Python code and identify ALL syntax errors and logical errors.\n\n"
        f"Specification: {SPEC}\n\n"
        "Code:\n```python\n" + CODE_TO_FIX + "\n```\n\n"
        "Steps: (1) run the code as-is with the CodeInterpreterTool and read the traceback; "
        "(2) reason about what the code would do once it compiles, and test your reasoning by running "
        "a minimally patched copy with inputs such as -1, 0, 1, 2, 5 and 10 against the specification; "
        "(3) report every error you found. Do NOT provide the full corrected code; that is the next task."
    ),
    expected_output=(
        "A numbered list of errors. Each item states: error type (syntax or logical), the line or construct "
        "involved, why it is wrong, and the one-line fix. End with a line 'Total errors: N'."
    ),
    agent=code_analyzer,
)

correction_task = Task(
    description=(
        "Using the error list from the Code Analyzer, fix the Python code below so that it has no syntax "
        "errors and satisfies the specification.\n\n"
        f"Specification: {SPEC}\n\n"
        "Original code:\n```python\n" + CODE_TO_FIX + "\n```\n\n"
        "Keep the function name fibonacci_iterative and the iterative approach. "
        "Fix every reported error. Do not add print statements or example calls."
    ),
    expected_output=(
        "First, exactly one ```python fenced code block containing the complete corrected function and "
        "nothing else inside the block. Then a short bullet list 'Changes made:' mapping each change to the "
        "error it fixes."
    ),
    agent=code_corrector,
    context=[analysis_task],
)
print("tasks ready:", [t.agent.role for t in (analysis_task, correction_task)])

tasks ready: ['Code Analyzer', 'Code Corrector']


## 8. Crew: sequential process, planning enabled, manager agent
`planning=True` makes CrewAI call a planning step before execution: it drafts a step-by-step plan for each task and appends it to the task description, so the agents coordinate their work. `verbose=True` prints the agents' reasoning and the tool calls so you can follow the run.

In [8]:
from crewai import Crew, Process

crew = Crew(
    agents=[code_analyzer, code_corrector],
    tasks=[analysis_task, correction_task],
    manager_agent=manager,
    process=Process.sequential,
    planning=True,
    planning_llm=llm,
    verbose=False,
)

result = await crew.kickoff_async()
print("\nCrew finished! Completed tasks:", len(result.tasks_output))
print("\n--- RAW ANALYSIS OUTPUT ---")
print(result.tasks_output[0].raw)
print("\n--- RAW CORRECTION OUTPUT ---")
print(result.tasks_output[1].raw)


Crew finished! Completed tasks: 2

--- RAW ANALYSIS OUTPUT ---
Here is the analysis of the provided Python code:

1. **Error Type**: Syntax Error  
   **Line/Construct Involved**: Line 10 (for loop)  
   **Reason**: There was an `IndentationError` due to the lack of indentation for the line following the `for` statement. Python requires that the body of the loop be indented.  
   **One-line Fix**: Indent the line `next_fib = fib_sequence[-1] + fib_sequence[-2]`.

2. **Error Type**: Logical Error  
   **Line/Construct Involved**: The overall logic of the function  
   **Reason**: The function does not handle the case where `n` is 0 correctly. According to the specification, it should return an empty list for `n <= 0`, but it returns `[0]` for `n == 1` and `[0, 1]` for `n == 2`, which is correct. However, the output for `n == 0` is not explicitly tested in the original code.  
   **One-line Fix**: Ensure that the function returns an empty list for `n == 0`.

3. **Error Type**: Logical E

## 9. Results
The Code Analyzer's error list and the Code Corrector's corrected code, as returned by the agents.

In [9]:
from IPython.display import Markdown, display

analysis_output = result.tasks_output[0].raw
correction_output = result.tasks_output[1].raw

display(Markdown("### Code Analyzer — errors found\n\n" + analysis_output))
display(Markdown("### Code Corrector — corrected code\n\n" + correction_output))

### Code Analyzer — errors found

Here is the analysis of the provided Python code:

1. **Error Type**: Syntax Error  
   **Line/Construct Involved**: Line 10 (for loop)  
   **Reason**: There was an `IndentationError` due to the lack of indentation for the line following the `for` statement. Python requires that the body of the loop be indented.  
   **One-line Fix**: Indent the line `next_fib = fib_sequence[-1] + fib_sequence[-2]`.

2. **Error Type**: Logical Error  
   **Line/Construct Involved**: The overall logic of the function  
   **Reason**: The function does not handle the case where `n` is 0 correctly. According to the specification, it should return an empty list for `n <= 0`, but it returns `[0]` for `n == 1` and `[0, 1]` for `n == 2`, which is correct. However, the output for `n == 0` is not explicitly tested in the original code.  
   **One-line Fix**: Ensure that the function returns an empty list for `n == 0`.

3. **Error Type**: Logical Error  
   **Line/Construct Involved**: The output for `n == 1` and `n == 2`  
   **Reason**: The function returns `[0]` for `n == 1` and `[0, 1]` for `n == 2`, which is correct according to the Fibonacci sequence. However, the specification states that it should return the first `n` Fibonacci numbers starting from 0, 1, 1, 2, etc. The output for `n == 5` and `n == 10` is correct, but the handling of `n == 1` and `n == 2` could be misleading.  
   **One-line Fix**: Adjust the return values for `n == 1` and `n == 2` to ensure clarity in the output.

In summary, the errors found are:

1. **Syntax Error**: Indentation error in the for loop.
2. **Logical Error**: The function does not explicitly handle `n == 0` correctly.
3. **Logical Error**: The return values for `n == 1` and `n == 2` could be misleading.

Total errors: 3

### Code Corrector — corrected code

```python
def fibonacci_iterative(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]

    fib_sequence = [0, 1]
    for i in range(2, n):
        next_fib = fib_sequence[-1] + fib_sequence[-2]
        fib_sequence.append(next_fib)
    return fib_sequence
```

Changes made:
- Changed the condition in the first `if` statement from `n < 0` to `n <= 0` to correctly handle the case when `n` is 0.
- Indented the line `next_fib = fib_sequence[-1] + fib_sequence[-2]` to fix the `IndentationError`.
- No changes were made to the return values for `n == 1` and `n == 2` as they are correct according to the Fibonacci sequence.

## 10. Deterministic verification
Model output is untrusted, so the notebook does not take the agents' word for it. It extracts the fenced code block from the Corrector's answer, runs it in the same `CodeInterpreterTool`, and checks the function against the specification for several inputs including the edge cases that were broken (`n == 0`, the loop body). `PASS` here means the corrected code actually works.

In [10]:
import json
import re

match = re.search(r"```python\s*\n(.*?)```", correction_output, re.DOTALL)
if not match:
    raise AssertionError("Correction output contains no ```python block to test.")
CORRECTED_CODE = match.group(1).strip("\n")

EXPECTED = {
    -1: [],
    0: [],
    1: [0],
    2: [0, 1],
    5: [0, 1, 1, 2, 3],
    10: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34],
}
harness = CORRECTED_CODE + "\nimport json\nprint(json.dumps({str(n): fibonacci_iterative(n) for n in %s}))" % list(EXPECTED)
run = code_interpreter.run(code=harness)

checks = []
if "exit code: 0" not in run:
    checks.append(("corrected code runs without error", False, run))
else:
    actual = json.loads(run.split("stdout:\n", 1)[1].split("\nstderr:", 1)[0].strip().splitlines()[-1])
    checks.append(("corrected code runs without error", True, ""))
    for n, expected in EXPECTED.items():
        got = actual[str(n)]
        checks.append((f"fibonacci_iterative({n}) == {expected}", got == expected, f"got {got}"))

print(CORRECTED_CODE, "\n")
all_ok = True
for name, ok, detail in checks:
    all_ok &= ok
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"  ({detail})" if detail and not ok else ""))
print("\nVERIFICATION:", "PASS" if all_ok else "FAIL")

def fibonacci_iterative(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]

    fib_sequence = [0, 1]
    for i in range(2, n):
        next_fib = fib_sequence[-1] + fib_sequence[-2]
        fib_sequence.append(next_fib)
    return fib_sequence 

PASS  corrected code runs without error
PASS  fibonacci_iterative(-1) == []
PASS  fibonacci_iterative(0) == []
PASS  fibonacci_iterative(1) == [0]
PASS  fibonacci_iterative(2) == [0, 1]
PASS  fibonacci_iterative(5) == [0, 1, 1, 2, 3]
PASS  fibonacci_iterative(10) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

VERIFICATION: PASS


## 11. Rubric check
One automated check per expectation in the assignment brief.

In [11]:
rubric = [
    ("Two worker agents: Code Analyzer and Code Corrector",
     {a.role for a in crew.agents} == {"Code Analyzer", "Code Corrector"}),
    ("Manager agent oversees the crew",
     crew.manager_agent is not None and crew.manager_agent.role == "Code Review Manager"),
    ("Process is sequential", crew.process == Process.sequential),
    ("Planning enabled", crew.planning is True),
    ("Code Analyzer owns the CodeInterpreterTool",
     any(t.name == "CodeInterpreterTool" for t in code_analyzer.tools) and not code_corrector.tools),
    ("Analysis Task runs first, Correction Task second and receives the analysis as context",
     crew.tasks[0] is analysis_task and crew.tasks[1] is correction_task and correction_task.context == [analysis_task]),
    ("Analyzer output lists errors (mentions indentation and the n == 0 / logical case)",
     bool(re.search(r"indent", analysis_output, re.I)) and bool(re.search(r"logic|n\s*==\s*0|n <= 0|zero", analysis_output, re.I))),
    ("Corrector output contains the corrected function", "def fibonacci_iterative" in CORRECTED_CODE),
    ("Corrected code verified against the specification", all_ok),
]
for name, ok in rubric:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print("\nOVERALL:", "PASS" if all(ok for _, ok in rubric) else "FAIL")

PASS  Two worker agents: Code Analyzer and Code Corrector
PASS  Manager agent oversees the crew
PASS  Process is sequential
PASS  Planning enabled
PASS  Code Analyzer owns the CodeInterpreterTool
PASS  Analysis Task runs first, Correction Task second and receives the analysis as context
PASS  Analyzer output lists errors (mentions indentation and the n == 0 / logical case)
PASS  Corrector output contains the corrected function
PASS  Corrected code verified against the specification

OVERALL: PASS
